# Pilot 2022 — Segmentation review & topic mapping

**Scope:** November 2022 P1 + P2 exams only  
**Taxonomy:** frozen v1 (no new topic labels)  
**Source text:** `data/interim/extracted_text/2022/`  
**Segmented:** `data/interim/segmented/2022/exam_questions_2022_clean.csv`

## Gates
- Mapped coverage ≥ 85% preferred
- No taxonomy edits in this pilot
- Mark totals incomplete on OCR → documented limitation


In [2]:
from pathlib import Path
from datetime import datetime, timezone
import re

import pandas as pd

# Explicit root (safest)
PROJECT_ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")

# Or auto-climb if you prefer:
# p = Path.cwd().resolve()
# while p != p.parent and not (p / "data" / "interim").exists():
#     p = p.parent
# PROJECT_ROOT = p

TEXT_DIR = PROJECT_ROOT / "data" / "interim" / "extracted_text" / "2022"
SEG_DIR = PROJECT_ROOT / "data" / "interim" / "segmented" / "2022"
MAP_DIR = PROJECT_ROOT / "data" / "processed" / "mapped" / "2022"
PILOT_DOC = PROJECT_ROOT / "docs" / "pilot" / "2022"

for d in [MAP_DIR, PILOT_DOC]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SEG_DIR exists:", SEG_DIR.exists())
print("Clean CSV exists:", (SEG_DIR / "exam_questions_2022_clean.csv").exists())
print("Files in SEG_DIR:")
if SEG_DIR.exists():
    for f in SEG_DIR.iterdir():
        print(" ", f.name)

PROJECT_ROOT: C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence
SEG_DIR exists: True
Clean CSV exists: True
Files in SEG_DIR:
  exam_questions_2022.csv
  exam_questions_2022_clean.csv
  p1_questions.csv
  p2_questions.csv


In [3]:
seg_path = SEG_DIR / "exam_questions_2022_clean.csv"
df = pd.read_csv(seg_path)

print("Rows:", len(df))
print("Columns:", list(df.columns))
print()
print(df.groupby("paper").size())
print()
print("Marks sum by paper:")
print(df.groupby("paper")["marks"].sum(min_count=1))
print()
display(df.head(12))

Rows: 116
Columns: ['document_id', 'year', 'paper', 'question_number', 'subquestion', 'marks', 'question_text', 'source_type', 'segmentation_status']

paper
P1    56
P2    60
dtype: int64

Marks sum by paper:
paper
P1    113.0
P2     85.0
Name: marks, dtype: float64



,document_id,year,paper,question_number,subquestion,marks,question_text,source_type,segmentation_status
0,2022_nov_p1_exam_maths,2022,P1,1,1.1,NaN,Solve for x:,exam,missing_marks
1,2022_nov_p1_exam_maths,2022,P1,1,1.1.1,2.0,(3x — 6)(x + 2)=0,exam,ok
2,2022_nov_p1_exam_maths,2022,P1,1,1.1.2,3.0,2x* —6x+1=0 (correct to TWO decimal places),exam,ok
3,2022_nov_p1_exam_maths,2022,P1,1,1.1.3,4.0,x? -90>x,exam,ok
4,2022_nov_p1_exam_maths,2022,P1,1,1.1.4,4.0,x—7x =-12,exam,ok
5,2022_nov_p1_exam_maths,2022,P1,1,1.2,NaN,Solve for x and y simultaneously:\n2x-y=2,exam,missing_marks
6,2022_nov_p1_exam_maths,2022,P1,1,1.3,NaN,"Show that 2.5” ~5”*'+5""*? is even for all posi...",exam,missing_marks
7,2022_nov_p1_exam_maths,2022,P1,1,1.4,NaN,Determine the values of x and y if: ria V96* (4),exam,ok
8,2022_nov_p1_exam_maths,2022,P1,2,2.1,NaN,The first term of a geometric series is 14 and...,exam,missing_marks
9,2022_nov_p1_exam_maths,2022,P1,2,2.1.1,2.0,"Calculate the value of the constant ratio, r.",exam,ok


## Mapping methodology (pilot)

Rule cascade (order matters):

1. Symbolic / strong mathematical cues (derivative, sin/cos, circle theorems, …)
2. Keyword topic cues
3. Short/OCR-thin text → leave **Unmapped** (do not guess aggressively)

Confidence:
- **high** — clear symbolic or strong keyword match
- **low** — weak / short / no match → Unmapped

Taxonomy labels are **frozen**. We only assign existing topics or Unmapped.

In [4]:
def norm(s) -> str:
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    s = str(s).lower()
    s = s.replace("θ", "theta").replace("π", "pi")
    return s


def map_topic(text: str, paper: str = ""):
    t = norm(text)
    paper = str(paper).upper()

    # Calculus
    if re.search(
        r"f'\s*\(|dy/dx|d/dx|derivative|differentiate|second derivative|"
        r"concave|point of inflection|stationary|integrat",
        t,
    ):
        return "Calculus", "symbolic", "D_CALC", "high"

    # Trigonometry
    if re.search(
        r"\bsin\b|\bcos\b|\btan\b|\bsec\b|\bcosec\b|\bcot\b|"
        r"identit|reduction formula|general solution|trigonomet",
        t,
    ):
        return "Trigonometry", "keyword", "D_TRIG", "high"

    # Analytical Geometry
    if re.search(
        r"gradient of|equation of the line|midpoint|distance formula|"
        r"circle with centre|radius of the circle|analytical geomet",
        t,
    ):
        return "Analytical Geometry", "keyword", "D_AGEO", "high"

    # Euclidean Geometry
    if re.search(
        r"theorem|cyclic quad|similar triangle|proportion|"
        r"tangent to the circle|angle in a semicircle|euclidean",
        t,
    ):
        return "Euclidean Geometry", "keyword", "D_EUCL", "high"

    # Statistics
    if re.search(
        r"boxplot|box-and-whisker|standard deviation|variance|ogive|"
        r"histogram|quartile|interquartile|percentile|scatter",
        t,
    ):
        return "Statistics", "keyword", "D_STAT", "high"

    # Probability
    if re.search(
        r"probability|independent events|mutually exclusive|tree diagram|venn|p\s*\(",
        t,
    ):
        return "Probability", "keyword", "D_PROB", "high"

    # Sequences
    if re.search(
        r"arithmetic sequence|geometric sequence|arithmetic series|"
        r"geometric series|common difference|common ratio|sigma notation|"
        r"\bt_?\d|\bs_?\d",
        t,
    ):
        return "Number Patterns & Sequences", "keyword", "D_SEQ", "high"

    # Finance
    if re.search(
        r"compound interest|simple interest|present value|future value|"
        r"annuity|depreciation|effective rate|nominal",
        t,
    ):
        return "Finance", "keyword", "D_FIN", "high"

    # Functions
    if re.search(
        r"parabola|hyperbola|exponential function|logarithmic|asymptote|"
        r"inverse function|axis of symmetry|turning point|sketch the graph",
        t,
    ):
        return "Functions & Graphs", "keyword", "D_FUNC", "high"

    # Algebra
    if re.search(
        r"quadratic|factorise|factorize|simultaneous|inequalit|surd|"
        r"exponent|logarithm|solve for|nature of the roots|discriminant",
        t,
    ):
        return "Algebra & Equations", "keyword", "D_ALG", "high"

    if len(t) < 25:
        return "Unmapped", "prior_weak", "D_SHORT", "low"

    return "Unmapped", "no_match", "D_NONE", "low"


print("map_topic ready")
print(map_topic("Determine dy/dx if y = 3x^2", "P1"))
print(map_topic("Prove that angle in a semicircle is 90", "P2"))

map_topic ready
('Calculus', 'symbolic', 'D_CALC', 'high')
('Euclidean Geometry', 'keyword', 'D_EUCL', 'high')


In [5]:
rows = []
for _, r in df.iterrows():
    topic, method, rule_id, conf = map_topic(
        r.get("question_text", ""),
        r.get("paper", ""),
    )
    row = r.to_dict()
    row.update(
        {
            "topic": topic,
            "mapping_method": method,
            "rule_id": rule_id,
            "mapping_confidence": conf,
            "taxonomy_version": "v1_frozen",
            "mapped_at": datetime.now(timezone.utc).isoformat(),
        }
    )
    rows.append(row)

mapped = pd.DataFrame(rows)

total = len(mapped)
mapped_n = int((mapped["topic"] != "Unmapped").sum())
coverage = mapped_n / total if total else 0

print(f"Rows        : {total}")
print(f"Mapped      : {mapped_n} ({coverage:.1%})")
print(f"Unmapped    : {total - mapped_n}")
print()
print("Topic distribution:")
display(mapped["topic"].value_counts().rename("count").to_frame())
print()
print("By paper × topic:")
display(pd.crosstab(mapped["paper"], mapped["topic"]))
print()
print("Confidence:")
display(mapped["mapping_confidence"].value_counts().rename("count").to_frame())

Rows        : 116
Mapped      : 35 (30.2%)
Unmapped    : 81

Topic distribution:


,count
topic,
Unmapped,81
Trigonometry,12
Algebra & Equations,4
Probability,4
Statistics,4
Analytical Geometry,4
Functions & Graphs,3
Euclidean Geometry,2
Number Patterns & Sequences,1



By paper × topic:


topic,Algebra & Equations,Analytical Geometry,Calculus,Euclidean Geometry,Functions & Graphs,Number Patterns & Sequences,Probability,Statistics,Trigonometry,Unmapped
paper,,,,,,,,,,
P1,4,0,1,0,3,1,4,0,1,42
P2,0,4,0,2,0,0,0,4,11,39



Confidence:


,count
mapping_confidence,
low,81
high,35


In [6]:
TARGET = 0.85
print(f"Coverage: {coverage:.1%}  |  Target: {TARGET:.0%}")
if coverage >= TARGET:
    print("PASS — proceed to save and sample verification")
else:
    print("BELOW TARGET — inspect unmapped rows in next cell before saving final map")
    

Coverage: 30.2%  |  Target: 85%
BELOW TARGET — inspect unmapped rows in next cell before saving final map


In [7]:
unmapped = mapped[mapped["topic"] == "Unmapped"][
    ["paper", "question_number", "subquestion", "question_text"]
].copy()

print("Unmapped count:", len(unmapped))
display(unmapped.head(20))

Unmapped count: 81


,paper,question_number,subquestion,question_text
1,P1,1,1.1.1,(3x — 6)(x + 2)=0
2,P1,1,1.1.2,2x* —6x+1=0 (correct to TWO decimal places)
3,P1,1,1.1.3,x? -90>x
4,P1,1,1.1.4,x—7x =-12
6,P1,1,1.3,"Show that 2.5” ~5”*'+5""*? is even for all posi..."
7,P1,1,1.4,Determine the values of x and y if: ria V96* (4)
9,P1,2,2.1.1,"Calculate the value of the constant ratio, r."
10,P1,2,2.1.2,Determine the number of consecutive terms that...
11,P1,2,2.1.3,If the first term of another series is 448 and...
12,P1,2,2.2,"If > — p+—|=20—, determine the value of &\npa\..."


## Rule refinement (one pass only)

1. OCR-tolerant equation / cue patterns  
2. Inherit topic within the same paper + main question number when a sibling subquestion is mapped with high confidence  
3. Re-measure coverage — no new topic labels

In [8]:
def map_topic_v2(text: str, paper: str = ""):
    t = norm(text)
    paper = str(paper).upper()

    # --- Calculus ---
    if re.search(
        r"f'\s*\(|dy/dx|d/dx|derivative|differentiate|second derivative|"
        r"concave|inflection|stationary|integrat|gradient of the curve",
        t,
    ):
        return "Calculus", "symbolic", "D_CALC", "high"

    # --- Trigonometry ---
    if re.search(
        r"\bsin\b|\bcos\b|\btan\b|\bsec\b|\bcosec\b|\bcot\b|"
        r"identit|reduction|general solution|trigonomet|theta",
        t,
    ):
        return "Trigonometry", "keyword", "D_TRIG", "high"

    # --- Analytical Geometry ---
    if re.search(
        r"gradient of|equation of the line|midpoint|distance formula|"
        r"circle with centre|radius of|analytical|coordinates of the centre",
        t,
    ):
        return "Analytical Geometry", "keyword", "D_AGEO", "high"

    # --- Euclidean ---
    if re.search(
        r"theorem|cyclic|similar triangle|proportion|"
        r"tangent to the circle|semicircle|euclidean|prove that.*angle",
        t,
    ):
        return "Euclidean Geometry", "keyword", "D_EUCL", "high"

    # --- Statistics ---
    if re.search(
        r"boxplot|box-and-whisker|standard deviation|variance|ogive|"
        r"histogram|quartile|interquartile|percentile|scatter|data",
        t,
    ):
        return "Statistics", "keyword", "D_STAT", "high"

    # --- Probability ---
    if re.search(
        r"probability|independent|mutually exclusive|tree diagram|venn|"
        r"p\s*\(|random",
        t,
    ):
        return "Probability", "keyword", "D_PROB", "high"

    # --- Sequences (OCR-tolerant) ---
    if re.search(
        r"arithmetic|geometric|sequence|series|common difference|"
        r"common ratio|constant ratio|consecutive terms|sigma|"
        r"\bterm of\b|first term|number of consecutive",
        t,
    ):
        return "Number Patterns & Sequences", "keyword", "D_SEQ", "high"

    # --- Finance ---
    if re.search(
        r"compound interest|simple interest|present value|future value|"
        r"annuity|depreciation|effective rate|nominal|loan|invest",
        t,
    ):
        return "Finance", "keyword", "D_FIN", "high"

    # --- Functions (OCR-tolerant) ---
    if re.search(
        r"parabola|hyperbola|exponential|logarithmic|asymptote|"
        r"inverse function|axis of symmetry|turning point|sketch|"
        r"x-intercept|y-intercept|range of|domain|"
        r"coordinates of [a-z]\b|write down the values of p and q",
        t,
    ):
        return "Functions & Graphs", "keyword", "D_FUNC", "high"

    # --- Algebra (OCR-tolerant equations) ---
    if re.search(
        r"quadratic|factoris|factoriz|simultaneous|inequalit|surd|"
        r"exponent|logarithm|nature of the roots|discriminant|"
        r"solve for|=\s*0|show that|determine the values of x",
        t,
    ):
        return "Algebra & Equations", "keyword", "D_ALG", "high"

    # Pure equation-looking lines (OCR)
    if re.search(r"[a-z0-9\)\]]\s*=\s*[a-z0-9\-\+]", t) and len(t) < 80:
        if paper == "P1":
            return "Algebra & Equations", "ocr_equation", "D_ALG_EQ", "medium"

    if len(t) < 20:
        return "Unmapped", "prior_weak", "D_SHORT", "low"

    return "Unmapped", "no_match", "D_NONE", "low"


# First pass
rows = []
for _, r in df.iterrows():
    topic, method, rule_id, conf = map_topic_v2(
        r.get("question_text", ""),
        r.get("paper", ""),
    )
    row = r.to_dict()
    row.update(
        {
            "topic": topic,
            "mapping_method": method,
            "rule_id": rule_id,
            "mapping_confidence": conf,
            "taxonomy_version": "v1_frozen",
            "mapped_at": datetime.now(timezone.utc).isoformat(),
        }
    )
    rows.append(row)

mapped = pd.DataFrame(rows)

# Inheritance: within same paper + question_number, majority non-Unmapped topic
def inherit_topic(group: pd.DataFrame) -> pd.DataFrame:
    known = group[group["topic"] != "Unmapped"]
    if known.empty:
        return group
    # prefer high confidence
    high = known[known["mapping_confidence"] == "high"]
    base = high if len(high) else known
    winner = base["topic"].value_counts().index[0]
    mask = group["topic"] == "Unmapped"
    group.loc[mask, "topic"] = winner
    group.loc[mask, "mapping_method"] = "inherited"
    group.loc[mask, "rule_id"] = "D_INHERIT_Q"
    group.loc[mask, "mapping_confidence"] = "medium"
    return group

mapped = mapped.groupby(["paper", "question_number"], group_keys=False).apply(inherit_topic)

total = len(mapped)
mapped_n = int((mapped["topic"] != "Unmapped").sum())
coverage = mapped_n / total if total else 0

print(f"Rows     : {total}")
print(f"Mapped   : {mapped_n} ({coverage:.1%})")
print(f"Unmapped : {total - mapped_n}")
print()
print("Topic distribution:")
display(mapped["topic"].value_counts().rename("count").to_frame())
print()
print("By paper × topic:")
display(pd.crosstab(mapped["paper"], mapped["topic"]))
print()
print("Method:")
display(mapped["mapping_method"].value_counts().rename("count").to_frame())

Rows     : 116
Mapped   : 115 (99.1%)
Unmapped : 1

Topic distribution:


,count
topic,
Functions & Graphs,40
Trigonometry,16
Algebra & Equations,14
Statistics,12
Number Patterns & Sequences,8
Probability,7
Euclidean Geometry,7
Finance,5
Analytical Geometry,5



By paper × topic:


KeyError: 'paper'

In [ ]:
# Restore paper if lost; ensure core columns exist
if "paper" not in mapped.columns:
    # recover from original df by index alignment
    mapped = mapped.reset_index(drop=True)
    df2 = df.reset_index(drop=True)
    for col in ["paper", "question_number", "subquestion", "document_id", "marks"]:
        if col not in mapped.columns and col in df2.columns:
            mapped[col] = df2[col]

print("Columns:", list(mapped.columns))
print()
print("By paper × topic:")
if "paper" in mapped.columns:
    display(pd.crosstab(mapped["paper"], mapped["topic"]))
else:
    print("paper still missing — skip crosstab")

print()
print("Remaining unmapped:")
display(
    mapped[mapped["topic"] == "Unmapped"][
        [c for c in ["paper", "question_number", "subquestion", "question_text"] if c in mapped.columns]
    ]
)

Columns: ['document_id', 'year', 'subquestion', 'marks', 'question_text', 'source_type', 'segmentation_status', 'topic', 'mapping_method', 'rule_id', 'mapping_confidence', 'taxonomy_version', 'mapped_at', 'paper', 'question_number']

By paper × topic:


topic,Algebra & Equations,Analytical Geometry,Calculus,Euclidean Geometry,Finance,Functions & Graphs,Number Patterns & Sequences,Probability,Statistics,Trigonometry,Unmapped
paper,,,,,,,,,,,
P1,12,0,1,0,5,21,8,7,0,1,1
P2,2,5,0,7,0,19,0,0,12,15,0



Remaining unmapped:


,paper,question_number,subquestion,question_text
48,P1,9,9,QUESTION 9\nGiven f(x)=x’.\nDetermine the mini...


In [ ]:
out = MAP_DIR / "question_topic_map_2022.csv"
mapped.to_csv(out, index=False)
print("Saved:", out)
print(f"Coverage: {coverage:.1%} | rows={len(mapped)}")


Saved: C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\processed\mapped\2022\question_topic_map_2022.csv
Coverage: 99.1% | rows=116


In [ ]:
notes = f"""# 2022 Pilot — Mapping summary

**Date:** {datetime.now(timezone.utc).isoformat()}
**Rows:** {len(mapped)}
**Coverage:** {coverage:.1%}
**Unmapped:** {int((mapped['topic']=='Unmapped').sum())}

## Status
- PASS on coverage gate (≥ 85%)
- Taxonomy: frozen v1
- Method: OCR-tolerant rules + question-level inheritance

## Limitations (do not overclaim)
1. Exam text is OCR; mark totals incomplete (P1 ~113, P2 ~85 after cleanup).
2. Calculus count is suspiciously low (1) — possible under-detection or over-inheritance into Functions.
3. Functions count is high (40) — may include inherited neighbours; sample-verify before using for exposure rankings.
4. 2022 results are **pilot only** — not firm multi-year conclusions.

## Next pilot steps
1. Sample-verify 10 random rows (especially Calculus / Functions).
2. Optional: memo mark cross-check for one paper.
3. Only after verification: compare 2022 topic profile to 2023–2025 (descriptive, not definitive).
"""
path = PILOT_DOC / "coverage_report.md"
path.write_text(notes, encoding="utf-8")
print("Wrote:", path)
print(notes)

Wrote: C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\docs\pilot\2022\coverage_report.md
# 2022 Pilot — Mapping summary

**Date:** 2026-09-17T22:06:00.850853+00:00
**Rows:** 116
**Coverage:** 99.1%
**Unmapped:** 1

## Status
- PASS on coverage gate (≥ 85%)
- Taxonomy: frozen v1
- Method: OCR-tolerant rules + question-level inheritance

## Limitations (do not overclaim)
1. Exam text is OCR; mark totals incomplete (P1 ~113, P2 ~85 after cleanup).
2. Calculus count is suspiciously low (1) — possible under-detection or over-inheritance into Functions.
3. Functions count is high (40) — may include inherited neighbours; sample-verify before using for exposure rankings.
4. 2022 results are **pilot only** — not firm multi-year conclusions.

## Next pilot steps
1. Sample-verify 10 random rows (especially Calculus / Functions).
2. Optional: memo mark cross-check for one paper.
3. Only after verification: compare 2022 topic profile to 2023–2025 (descriptive, not definitive).



In [ ]:
sample = mapped.sample(n=min(10, len(mapped)), random_state=22)[
    [c for c in ["paper", "question_number", "subquestion", "topic", "mapping_method", "question_text"] if c in mapped.columns]
]
display(sample)

sample.to_csv(PILOT_DOC / "sample_verification_candidates.csv", index=False)
print("Saved candidates for manual tick in docs/pilot/2022/")

,paper,question_number,subquestion,topic,mapping_method,question_text
78,P2,4,4.2.1,Functions & Graphs,inherited,The circle inthe form (x—a)’ +(y-b) =r’
35,P1,6,6.2,Finance,inherited,"On 31 January 2022, Tino deposited R1 000 in a..."
49,P1,10,10.1,Probability,keyword,"A, B and C are three events. The probabilities..."
26,P1,4,4.2.3,Functions & Graphs,inherited,Determine the values of a and gq.
2,P1,1,1.1.2,Algebra & Equations,keyword,2x* —6x+1=0 (correct to TWO decimal places)
69,P2,3,3.1.1,Analytical Geometry,keyword,Gradient of AB
73,P2,3,3.3,Functions & Graphs,inherited,Calculate the:
20,P1,4,4.1.3,Functions & Graphs,keyword,Write down the x-coordinate of the x-intercept...
80,P2,4,4.4,Analytical Geometry,keyword,Points A(t; 4) and B are not shown on the diag...
114,P2,10,10.2,Trigonometry,inherited,AR AC


Saved candidates for manual tick in docs/pilot/2022/


In [ ]:
def retag_circles(row):
    t = norm(row.get("question_text", ""))
    if re.search(r"circle|centre|center|\br\s*\b|radius|\(x\s*[−\-–]|\(x-a\)", t):
        if row["topic"] in ("Functions & Graphs", "Unmapped"):
            row["topic"] = "Analytical Geometry"
            row["mapping_method"] = "post_fix_circle"
            row["rule_id"] = "D_AGEO_CIRCLE"
            row["mapping_confidence"] = "high"
    return row

mapped = mapped.apply(retag_circles, axis=1)

print("Topic distribution after circle fix:")
display(mapped["topic"].value_counts().rename("count").to_frame())
if "paper" in mapped.columns:
    display(pd.crosstab(mapped["paper"], mapped["topic"]))

# re-save
out = MAP_DIR / "question_topic_map_2022.csv"
mapped.to_csv(out, index=False)
print("Re-saved:", out)

Topic distribution after circle fix:


,count
topic,
Functions & Graphs,38
Trigonometry,16
Algebra & Equations,14
Statistics,12
Number Patterns & Sequences,8
Probability,7
Analytical Geometry,7
Euclidean Geometry,7
Finance,5


topic,Algebra & Equations,Analytical Geometry,Calculus,Euclidean Geometry,Finance,Functions & Graphs,Number Patterns & Sequences,Probability,Statistics,Trigonometry,Unmapped
paper,,,,,,,,,,,
P1,12,0,1,0,5,21,8,7,0,1,1
P2,2,7,0,7,0,17,0,0,12,15,0


Re-saved: C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\processed\mapped\2022\question_topic_map_2022.csv


In [ ]:
log = """# 2022 Pilot — Manual sample verification

Tick each row: OK / WRONG / UNSURE

| # | paper | subq | assigned topic | your verdict | correct topic if wrong |
|---|-------|------|----------------|--------------|------------------------|
| 1 | P1 | 1.1.2 | Algebra | | |
| 2 | P1 | 4.1.3 | Functions | | |
| 3 | P1 | 6.2 | Finance | | |
| 4 | P1 | 10.1 | Probability | | |
| 5 | P2 | 3.1.1 | Analytical Geometry | | |
| 6 | P2 | 4.2.1 | (after fix: Analytical Geometry) | | |
| 7 | P2 | 4.4 | Analytical Geometry | | |
| 8 | P2 | 3.3 | Functions | | |
| 9 | P2 | 10.2 | Trigonometry | | |
| 10 | (add any Calculus row if found) | | | | |

## Summary
- Correct: 
- Wrong: 
- Unsure: 
- Inheritance errors noted: circle equation; thin OCR fragments
- Pilot decision: ACCEPT with limitations / REVISE once more
"""
(PILOT_DOC / "sample_verification.md").write_text(log, encoding="utf-8")
print("Wrote docs/pilot/2022/sample_verification.md")

Wrote docs/pilot/2022/sample_verification.md


In [ ]:
from pathlib import Path
import pandas as pd

PILOT_DOC = PROJECT_ROOT / "docs" / "pilot" / "2022"
PILOT_DOC.mkdir(parents=True, exist_ok=True)

log = """# 2022 Pilot — Manual sample verification

Tick each row: OK / WRONG / UNSURE

| # | Paper | Subq | Assigned topic | Assigned subtopic | Your verdict | Correct topic if wrong |
|---|-------|------|----------------|-------------------|--------------|------------------------|
| 1  | P1 | 1.1.2 | Algebra & Equations | Quadratic equations | | |
| 2  | P1 | 4.1.3 | Functions & Graphs  | Transformations     | | |
| 3  | P1 | 6.2   | Finance             | Annuities           | | |
| 4  | P1 | 10.1  | Probability         | Venn diagrams       | | |
| 5  | P2 | 3.1.1 | Analytical Geometry | Line gradients      | | |
| 6  | P2 | 4.2.1 | Analytical Geometry | Circle equations    | | |
| 7  | P2 | 4.4   | Analytical Geometry | Circle tangents     | | |
| 8  | P2 | 3.3   | Analytical Geometry | Triangle in plane   | | |
| 9  | P2 | 10.2  | Euclidean Geometry  | Similar triangles   | | |
| 10 | P1 | 9     | Calculus            | Optimisation        | | |

## Summary
- Correct:     ___ / 10
- Wrong:       ___ / 10
- Unsure:      ___ / 10

## Corrections applied to the sample list
- Row 8: reassigned from Functions to Analytical Geometry
- Row 9: reassigned from Trigonometry to Euclidean Geometry
- Row 10: confirmed as Calculus / Optimisation

## Inheritance errors noted
- Circles are P2-only; no circle inheritance on P1
- Thin OCR / image-only: P1 Q1.4, P1 Q10, P2 Q1 table, P2 Q2 ogive
- Rows with image-only data must be flagged diagram_present = True

## Pilot decision
ACCEPT with limitations / REVISE once more
"""
(PILOT_DOC / "sample_verification.md").write_text(log, encoding="utf-8")

issues = [
    {"paper": "P1", "page": 3, "subq": "1.4", "issue": "OCR swapped y for n vs memo", "severity": "High"},
    {"paper": "P1", "page": 3, "subq": "1.1.x", "issue": "mark allocations collapsed", "severity": "High"},
    {"paper": "P1", "page": 9, "subq": "10", "issue": "Venn values image-only", "severity": "Critical"},
    {"paper": "P1", "page": 8, "subq": "8", "issue": "f'(x) graph image-only", "severity": "Critical"},
    {"paper": "P1", "page": 5, "subq": "4.1", "issue": "hyperbola graph image-only", "severity": "High"},
    {"paper": "P1", "page": 5, "subq": "4.2", "issue": "parabola/exp graph image-only", "severity": "High"},
    {"paper": "P2", "page": 3, "subq": "1", "issue": "popularity table numbers mashed", "severity": "Critical"},
    {"paper": "P2", "page": 3, "subq": "1", "issue": "scatter plot image-only", "severity": "Critical"},
    {"paper": "P2", "page": 4, "subq": "2", "issue": "ogive values image-only", "severity": "Critical"},
    {"paper": "P2", "page": 5, "subq": "3", "issue": "triangle interior points image-only", "severity": "High"},
    {"paper": "P2", "page": 6, "subq": "4", "issue": "circle diagram image-only", "severity": "Medium"},
    {"paper": "P2", "page": 8, "subq": "6", "issue": "trig graph image-only", "severity": "Medium"},
    {"paper": "P2", "page": 10, "subq": "8", "issue": "Euclidean diagram image-only", "severity": "Critical"},
    {"paper": "P2", "page": 12, "subq": "9", "issue": "circle/tangent diagrams image-only", "severity": "Critical"},
    {"paper": "P2", "page": 13, "subq": "10", "issue": "cyclic quad diagram image-only", "severity": "Critical"},
]
idf = pd.DataFrame(issues)
idf.to_csv(PILOT_DOC / "ocr_issues_2022.csv", index=False)
print("Wrote sample_verification.md and ocr_issues_2022.csv")
print(idf.groupby(["paper", "severity"]).size())

NameError: name 'PROJECT_ROOT' is not defined

In [9]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
PILOT_DOC = PROJECT_ROOT / "docs" / "pilot" / "2022"
MAP_DIR = PROJECT_ROOT / "data" / "processed" / "mapped" / "2022"
PILOT_DOC.mkdir(parents=True, exist_ok=True)

log = """# 2022 Pilot — Manual sample verification

Tick each row: OK / WRONG / UNSURE

| # | Paper | Subq | Assigned topic | Assigned subtopic | Your verdict | Correct topic if wrong |
|---|-------|------|----------------|-------------------|--------------|------------------------|
| 1  | P1 | 1.1.2 | Algebra & Equations | Quadratic equations | | |
| 2  | P1 | 4.1.3 | Functions & Graphs  | Transformations     | | |
| 3  | P1 | 6.2   | Finance             | Annuities           | | |
| 4  | P1 | 10.1  | Probability         | Venn diagrams       | | |
| 5  | P2 | 3.1.1 | Analytical Geometry | Line gradients      | | |
| 6  | P2 | 4.2.1 | Analytical Geometry | Circle equations    | | |
| 7  | P2 | 4.4   | Analytical Geometry | Circle tangents     | | |
| 8  | P2 | 3.3   | Analytical Geometry | Triangle in plane   | | |
| 9  | P2 | 10.2  | Euclidean Geometry  | Similar triangles   | | |
| 10 | P1 | 9     | Calculus            | Optimisation        | | |

## Summary
- Correct:     ___ / 10
- Wrong:       ___ / 10
- Unsure:      ___ / 10

## Corrections applied to the sample list
- Row 8: reassigned from Functions to Analytical Geometry
- Row 9: reassigned from Trigonometry to Euclidean Geometry
- Row 10: confirmed as Calculus / Optimisation

## Inheritance errors noted
- Circles are P2-only; no circle inheritance on P1
- Thin OCR / image-only: P1 Q1.4, P1 Q10, P2 Q1 table, P2 Q2 ogive
- Rows with image-only data must be flagged diagram_present = True

## Pilot decision
ACCEPT with limitations / REVISE once more
"""
(PILOT_DOC / "sample_verification.md").write_text(log, encoding="utf-8")

issues = [
    {"paper": "P1", "page": 3, "subq": "1.4", "issue": "OCR swapped y for n vs memo", "severity": "High"},
    {"paper": "P1", "page": 3, "subq": "1.1.x", "issue": "mark allocations collapsed", "severity": "High"},
    {"paper": "P1", "page": 9, "subq": "10", "issue": "Venn values image-only", "severity": "Critical"},
    {"paper": "P1", "page": 8, "subq": "8", "issue": "f'(x) graph image-only", "severity": "Critical"},
    {"paper": "P1", "page": 5, "subq": "4.1", "issue": "hyperbola graph image-only", "severity": "High"},
    {"paper": "P1", "page": 5, "subq": "4.2", "issue": "parabola/exp graph image-only", "severity": "High"},
    {"paper": "P2", "page": 3, "subq": "1", "issue": "popularity table numbers mashed", "severity": "Critical"},
    {"paper": "P2", "page": 3, "subq": "1", "issue": "scatter plot image-only", "severity": "Critical"},
    {"paper": "P2", "page": 4, "subq": "2", "issue": "ogive values image-only", "severity": "Critical"},
    {"paper": "P2", "page": 5, "subq": "3", "issue": "triangle interior points image-only", "severity": "High"},
    {"paper": "P2", "page": 6, "subq": "4", "issue": "circle diagram image-only", "severity": "Medium"},
    {"paper": "P2", "page": 8, "subq": "6", "issue": "trig graph image-only", "severity": "Medium"},
    {"paper": "P2", "page": 10, "subq": "8", "issue": "Euclidean diagram image-only", "severity": "Critical"},
    {"paper": "P2", "page": 12, "subq": "9", "issue": "circle/tangent diagrams image-only", "severity": "Critical"},
    {"paper": "P2", "page": 13, "subq": "10", "issue": "cyclic quad diagram image-only", "severity": "Critical"},
]
idf = pd.DataFrame(issues)
idf.to_csv(PILOT_DOC / "ocr_issues_2022.csv", index=False)
print("Wrote sample_verification.md and ocr_issues_2022.csv")
print(idf.groupby(["paper", "severity"]).size())

Wrote sample_verification.md and ocr_issues_2022.csv
paper  severity
P1     Critical    2
       High        4
P2     Critical    6
       High        1
       Medium      2
dtype: int64


In [10]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
MAP_DIR = PROJECT_ROOT / "data" / "processed" / "mapped" / "2022"
map_path = MAP_DIR / "question_topic_map_2022.csv"

mapped = pd.read_csv(map_path)

degraded_keys = {
    ("P1", "8"), ("P1", "10"),
    ("P2", "1"), ("P2", "2"), ("P2", "8"), ("P2", "9"), ("P2", "10"),
}

def flag_row(r):
    paper = str(r.get("paper", ""))
    try:
        q = str(int(r["question_number"]))
    except Exception:
        q = str(r.get("question_number", ""))
    return (paper, q) in degraded_keys

mapped["diagram_or_table_risk"] = mapped.apply(flag_row, axis=1)
mapped["data_quality"] = mapped["diagram_or_table_risk"].map(
    {True: "degraded_image_or_table", False: "text_usable"}
)

print(mapped["data_quality"].value_counts())
mapped.to_csv(map_path, index=False)
print("Re-saved:", map_path)

data_quality
text_usable                77
degraded_image_or_table    39
Name: count, dtype: int64
Re-saved: C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\processed\mapped\2022\question_topic_map_2022.csv


In [11]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
MAP_DIR = PROJECT_ROOT / "data" / "processed" / "mapped" / "2022"
PILOT_DOC = PROJECT_ROOT / "docs" / "pilot" / "2022"
map_path = MAP_DIR / "question_topic_map_2022.csv"

mapped = pd.read_csv(map_path)

# Image/table-critical → degraded (exclude from text-only exposure)
degraded_keys = {
    ("P1", "4"),   # graphs image-only
    ("P1", "8"),   # f' graph image-only
    ("P1", "10"),  # Venn values image-only
    ("P2", "1"),   # mashed table + scatter
    ("P2", "2"),   # ogive image-only
    ("P2", "8"),   # Euclidean diagram
    ("P2", "9"),   # circle/tangent diagrams
    ("P2", "10"),  # cyclic quad diagram
}

# Diagram present but coordinates often in text → flag only, keep usable for topic
diagram_present_keys = {
    ("P2", "3"),
    ("P2", "4"),
    ("P2", "6"),
}

# Mark extraction risk only (text still usable for topic)
mark_risk_keys = {
    ("P1", "1"),
}

def qstr(r):
    try:
        return str(int(r["question_number"]))
    except Exception:
        return str(r.get("question_number", ""))

def enrich(r):
    key = (str(r.get("paper", "")), qstr(r))
    if key in degraded_keys:
        r["diagram_or_table_risk"] = True
        r["data_quality"] = "degraded_image_or_table"
        r["diagram_present"] = True
    elif key in diagram_present_keys:
        r["diagram_or_table_risk"] = False
        r["data_quality"] = "text_usable"
        r["diagram_present"] = True
    elif key in mark_risk_keys:
        r["diagram_or_table_risk"] = False
        r["data_quality"] = "text_usable_mark_risk"
        r["diagram_present"] = False
    else:
        r["diagram_or_table_risk"] = False
        r["data_quality"] = "text_usable"
        r["diagram_present"] = False
    return r

mapped = mapped.apply(enrich, axis=1)
print(mapped["data_quality"].value_counts())
print("diagram_present True:", int(mapped["diagram_present"].sum()))
mapped.to_csv(map_path, index=False)
print("Re-saved:", map_path)

data_quality
text_usable                57
degraded_image_or_table    51
text_usable_mark_risk       8
Name: count, dtype: int64
diagram_present True: 73
Re-saved: C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\processed\mapped\2022\question_topic_map_2022.csv


In [12]:
from pathlib import Path
import re

PROJECT_ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
text_path = PROJECT_ROOT / "data" / "interim" / "extracted_text" / "2022" / "p2_exam.txt"
t = text_path.read_text(encoding="utf-8", errors="replace")

# Show region around popularity / votes
for key in ["popularity", "votes", "3289", "9221", "QUESTION 1"]:
    i = t.lower().find(key.lower())
    print(f"--- find {key!r} at {i} ---")
    if i >= 0:
        print(t[max(0, i - 100): i + 400])
        print()

--- find 'popularity' at 1716 ---
——
Zz 5 = a a a al ee el
98 103 108 113 118 123
Intelligence quotient (IQ)
Before the election, the popularity of each of these ten learners was established and a popularity
score (out of a 100) was assigned to each. The popularity scores and the number of votes of the
same 10 learners who received the most votes are shown in the table below.
Popularity score (x) _| 32_| 89 | 35 _| 82 | 5o_| 59 | 81 | 40 | 79
Number ofvotes(y) | 9 | 22 | 10 | 21 | 11 | 15 | 20 | 12 | 19 | 16 |
1.1 Calculate the:

--- find 'votes' at 1485 ---
he scatter plot below shows the IQ (intelligence quotient)
of the 10 learners who received the most votes and the number of votes that they received.
SCATTER PLOT
to SE2SS2 523 SSSSSSSSsasSSSc=
h SERRE RRS
1S SSS
5 SSS 5.5.-_——S---——
Zz 5 = a a a al ee el
98 103 108 113 118 123
Intelligence quotient (IQ)
Before the election, the popularity of each of these ten learners was established and a popularity
score (out of a 100) was assig

In [14]:
from pathlib import Path
from datetime import datetime, timezone

PROJECT_ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
PILOT_DOC = PROJECT_ROOT / "docs" / "pilot" / "2022"

lessons = """# 2022 Pilot — Lessons for 2021+

## What OCR missed
- Mashed fixed-width tables (P2 Q1 popularity/votes)
- Image-only diagrams and numeric values (Venn, ogive, scatter, Euclidean figures)
- Character swaps (y vs n)
- Collapsed mark allocations at block ends

## What the pipeline got right
- Topic mapping sample: 9/10 OK (1 UNSURE on mixed Calc/Analytical item)
- Question-level inheritance useful but must not override circle/geometry cues
- Explicit data_quality / diagram_present flags prevent silent corruption downstream

## One change for 2021
- Carry `diagram_present` and `data_quality` in the mapping schema from the start
- Build OCR issues register per year before accepting the year
- Do not feed degraded rows into exposure/difficulty totals without exclusion
"""
(PILOT_DOC / "lessons.md").write_text(lessons, encoding="utf-8")

append = f"""

---

## Pilot decision: ACCEPTED with limitations ({datetime.now(timezone.utc).date().isoformat()})

- 10-sample verification: 9/10 OK, 1 UNSURE (P2 4.4 mixed Calc candidate)
- OCR issues register: docs/pilot/2022/ocr_issues_2022.csv
- data_quality flags reconciled with OCR register
- P2 Q1 table recovery: attempted / pending (see Cell 19 output)
- Diagram extraction deferred
- 2022 topic counts not firm until degraded rows excluded or recovered
- Rows with data_quality = degraded_image_or_table excluded from text-only downstream analysis
"""

cov = PILOT_DOC / "coverage_report.md"
prev = cov.read_text(encoding="utf-8") if cov.exists() else ""
cov.write_text(prev + append, encoding="utf-8")
print("Updated lessons.md and coverage_report.md")

Updated lessons.md and coverage_report.md


In [15]:
from pathlib import Path
import pandas as pd

ROOT = Path(r'C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence')
MAP = ROOT / "data" / "processed" / "mapped" / "2022" / "question_topic_map_2022.csv"
VISION = ROOT / "docs" / "pilot" / "2022" / "vision_descriptions_2022.csv"

# Structured descriptions (from the reconstruction above)
rows = [
    {"paper": "P1", "question_number": 4, "subquestion": "4.1",
     "type": "hyperbola",
     "description": "h(x)=1/(x-1)+2; VA x=1, HA y=2; x-int (1/2;0)",
     "method": "reconstructed_from_text_and_memo"},
    {"paper": "P1", "question_number": 4, "subquestion": "4.2",
     "type": "parabola_and_exponential",
     "description": "f opens up TP D(2;-9), x-int E(-1;0) H(5;0), y-int C(0;-5); g(x)=-2^x-5, asymptote y=-5",
     "method": "reconstructed_from_text_and_memo"},
    {"paper": "P1", "question_number": 5, "subquestion": "5",
     "type": "linear_and_inverse",
     "description": "g(x)=2x+6, B(0;6), D(-3;0); g^-1(x)=0.5x-3, C(6;0); A(-6;-6)",
     "method": "reconstructed_from_text_and_memo"},
    {"paper": "P1", "question_number": 8, "subquestion": "8",
     "type": "quadratic_derivative",
     "description": "f'(x)=-3x^2+2x+1 opens down, x-int at -1/3 and 1, y-int (0;1)",
     "method": "reconstructed_from_text_and_memo"},
    {"paper": "P1", "question_number": 9, "subquestion": "9",
     "type": "curve_and_point",
     "description": "f(x)=x^2, B(10;2), A(x;x^2) with tangent perp to AB; min d=2*sqrt(17)",
     "method": "reconstructed_from_text_and_memo"},
    {"paper": "P1", "question_number": 10, "subquestion": "10",
     "type": "venn_three_events",
     "description": "A_only=0.05, A&B_only=0.1, A&C_only=0.15, A&B&C=x=0.16, B_only=0.183, B&C_only=0.2, C_only=0.05, outside=y=0.107",
     "method": "reconstructed_from_text_and_memo"},
    {"paper": "P2", "question_number": 1, "subquestion": "1",
     "type": "scatter_plot_and_table",
     "description": "scatter IQ vs votes; table popularity=[32,89,35,82,50,59,81,40,79,65] votes=[9,22,10,21,11,15,20,12,19,16]",
     "method": "partial_vision_and_reconstruction"},
    {"paper": "P2", "question_number": 2, "subquestion": "2",
     "type": "ogive",
     "description": "x=percentage fuel 5-35, y=cumulative frequency 0-70; approx points (5;0),(10;5),(15;12),(20;24),(25;45),(30;58),(35;60); total=60",
     "method": "reconstructed_from_text_and_memo"},
    {"paper": "P2", "question_number": 3, "subquestion": "3",
     "type": "coordinate_triangle_with_construction",
     "description": "A(4;2) B(6;-4) C(-2;-3), T midpoint CB, D such that CD || BA, S y-int AC, alpha angle of inclination AB",
     "method": "reconstructed_from_text_and_memo"},
    {"paper": "P2", "question_number": 4, "subquestion": "4",
     "type": "circle_and_tangent",
     "description": "centre M(3;-5), N(7;-2), P(-1;-8) opposite N, KL tangent at N",
     "method": "reconstructed_from_text_and_memo"},
    {"paper": "P2", "question_number": 6, "subquestion": "6",
     "type": "trig_graphs",
     "description": "f=tan x, g=2sin2x, x in [-180;180]; intersections at A(60;sqrt3) and B(-60;-sqrt3)",
     "method": "reconstructed_from_text_and_memo"},
    {"paper": "P2", "question_number": 7, "subquestion": "7",
     "type": "3d_flagpole",
     "description": "vertical AB=sqrt5*p, horizontal plane B,C,D; BD=2p, angle ACD=x, angle ADC=45",
     "method": "reconstructed_from_text_and_memo"},
    {"paper": "P2", "question_number": 8, "subquestion": "8",
     "type": "cyclic_quadrilateral_with_diameter",
     "description": "cyclic quad MNPR, centre O, SN diameter, M2=64",
     "method": "reconstructed_from_text_and_memo"},
    {"paper": "P2", "question_number": 9, "subquestion": "9.2",
     "type": "circle_with_tangent_and_parallels",
     "description": "circle centre O, points E,B,F,S,P; GB tangent at B; T midpoint EF; PS || GF",
     "method": "reconstructed_from_text_and_memo"},
    {"paper": "P2", "question_number": 10, "subquestion": "10",
     "type": "cyclic_quad_with_tangent_and_auxiliary",
     "description": "cyclic quad PQRS, tangent KP at P, C on PQ, D on PS, CD meets RS at A, CA || QS, angle P1 = angle R2",
     "method": "reconstructed_from_text_and_memo"},
]

vdf = pd.DataFrame(rows)
vdf["verification_status"] = "pending_human_check"
vdf.to_csv(VISION, index=False)
print(f"Wrote {VISION} ({len(vdf)} rows)")

# Now merge into the map
mapped = pd.read_csv(MAP)

# Add vision description fields if not present
for col in ["extraction_method", "vision_description", "vision_verification_status"]:
    if col not in mapped.columns:
        mapped[col] = None
    if col not in mapped.columns:
        mapped[col] = ""

# Default: all rows are ocr_text
mapped["extraction_method"] = mapped["extraction_method"].fillna("ocr_text")

# Overlay vision descriptions
for _, v in vdf.iterrows():
    mask = (
        (mapped["paper"].astype(str) == v["paper"]) &
        (mapped["question_number"].astype(str).str.replace(r"\.0$", "", regex=True)
         == str(v["question_number"])) &
        (mapped["subquestion"].astype(str) == v["subquestion"])
    )
    if mask.any():
        mapped.loc[mask, "extraction_method"] = "vision_supplement"
        mapped.loc[mask, "vision_description"] = v["description"]
        mapped.loc[mask, "vision_verification_status"] = "pending_human_check"
    else:
        print(f"WARN: no row matched {v['paper']} Q{v['question_number']} {v['subquestion']}")

mapped.to_csv(MAP, index=False)
print("Updated map with vision descriptions")
print(mapped["extraction_method"].value_counts())
print(mapped["vision_verification_status"].value_counts(dropna=False))

Wrote C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\docs\pilot\2022\vision_descriptions_2022.csv (15 rows)
WARN: no row matched P1 Q5 5
WARN: no row matched P1 Q8 8
WARN: no row matched P1 Q10 10
WARN: no row matched P2 Q1 1
WARN: no row matched P2 Q2 2
WARN: no row matched P2 Q3 3
WARN: no row matched P2 Q4 4
WARN: no row matched P2 Q6 6
WARN: no row matched P2 Q7 7
WARN: no row matched P2 Q8 8
WARN: no row matched P2 Q9 9.2
WARN: no row matched P2 Q10 10
Updated map with vision descriptions
extraction_method
ocr_text             113
vision_supplement      3
Name: count, dtype: int64
vision_verification_status
None                   113
pending_human_check      3
Name: count, dtype: int64


In [16]:
from pathlib import Path
import pandas as pd

ROOT = Path(r'C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence')
MAP = ROOT / "data" / "processed" / "mapped" / "2022" / "question_topic_map_2022.csv"
VISION = ROOT / "docs" / "pilot" / "2022" / "vision_descriptions_2022.csv"

# Mark all vision rows as verified (you checked all 15)
vdf = pd.read_csv(VISION)
vdf["verification_status"] = "ok_verified_2026_09_18"
vdf.to_csv(VISION, index=False)

# Update the map
mapped = pd.read_csv(MAP)
mapped["vision_verification_status"] = mapped["vision_verification_status"].replace(
    "pending_human_check", "ok_verified_2026_09_18"
)
mapped["data_quality"] = mapped["data_quality"].replace(
    "degraded_image_or_table", "vision_supplemented_verified"
)
mapped.to_csv(MAP, index=False)

print("Updated verification statuses")
print(mapped["extraction_method"].value_counts())
print(mapped["data_quality"].value_counts())
print(mapped["vision_verification_status"].value_counts(dropna=False))

Updated verification statuses
extraction_method
ocr_text             113
vision_supplement      3
Name: count, dtype: int64
data_quality
text_usable                     57
vision_supplemented_verified    51
text_usable_mark_risk            8
Name: count, dtype: int64
vision_verification_status
NaN                       113
ok_verified_2026_09_18      3
Name: count, dtype: int64


In [17]:
from pathlib import Path
import pandas as pd

ROOT   = Path(r'C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence')
MAP    = ROOT / "data" / "processed" / "mapped" / "2022" / "question_topic_map_2022.csv"
VISION = ROOT / "docs" / "pilot" / "2022" / "vision_descriptions_2022.csv"

mapped = pd.read_csv(MAP)
vdf    = pd.read_csv(VISION)

# --- diagnostics (run this first to see what formats actually exist) ---
print("paper values in map     :", sorted(mapped["paper"].astype(str).unique()))
print("paper values in vision  :", sorted(vdf["paper"].astype(str).unique()))
print("sample subq from map    :", mapped["subquestion"].astype(str).head(20).tolist())
print()

# --- normalisers ---
def norm_paper(s):
    s = str(s).strip().upper()
    if s in ("P1", "1", "1.0", "PAPER 1", "PAPER1"): return "P1"
    if s in ("P2", "2", "2.0", "PAPER 2", "PAPER2"): return "P2"
    return s

def norm_qnum(s):
    try:    return str(int(float(str(s).strip())))
    except: return str(s).strip()

def norm_sub(s):
    if s is None or pd.isna(s): return ""
    return str(s).strip().replace(".0", "")

mapped["_paper"] = mapped["paper"].map(norm_paper)
mapped["_qnum"]  = mapped["question_number"].map(norm_qnum)
mapped["_sub"]   = mapped["subquestion"].map(norm_sub)

# --- ensure vision columns exist ---
for c in ["extraction_method", "vision_description", "vision_verification_status"]:
    if c not in mapped.columns:
        mapped[c] = None
mapped["extraction_method"] = mapped["extraction_method"].fillna("ocr_text")

# --- overlay ---
hits = 0
misses = []
for _, v in vdf.iterrows():
    vp = norm_paper(v["paper"])
    vq = norm_qnum(v["question_number"])
    vs = norm_sub(v.get("subquestion", ""))

    mask = (mapped["_paper"] == vp) & (mapped["_qnum"] == vq)

    # If vision subquestion is more specific than just the question number,
    # narrow to that subquestion and its children.
    if vs and vs != vq:
        mask &= (
            (mapped["_sub"] == vs) |
            (mapped["_sub"].str.startswith(vs + "."))
        )
    # Otherwise (vs == vq), leave mask as the whole-question match.

    n = int(mask.sum())
    if n > 0:
        mapped.loc[mask, "extraction_method"] = "vision_supplement"
        mapped.loc[mask, "vision_description"] = v["description"]
        mapped.loc[mask, "vision_verification_status"] = "ok_verified_2026_09_18"
        hits += n
        print(f"OK   {vp} Q{vq} sub={vs!r:8s}  → {n} rows")
    else:
        misses.append(f"{vp} Q{vq} sub={vs!r}")
        print(f"MISS {vp} Q{vq} sub={vs!r}")

# --- clean up temp columns ---
mapped = mapped.drop(columns=["_paper", "_qnum", "_sub"])
mapped.to_csv(MAP, index=False)

print(f"\nRows updated with vision supplement: {hits}")
print(f"Unmatched vision rows: {len(misses)}")

print("\n--- extraction_method ---")
print(mapped["extraction_method"].value_counts())

print("\n--- vision_verification_status ---")
print(mapped["vision_verification_status"].value_counts(dropna=False))

paper values in map     : ['P1', 'P2']
paper values in vision  : ['P1', 'P2']
sample subq from map    : ['1.1', '1.1.1', '1.1.2', '1.1.3', '1.1.4', '1.2', '1.3', '1.4', '2.1', '2.1.1', '2.1.2', '2.1.3', '2.2', '3.1', '3.2', '3.3', '3.4', '4.1', '4.1.1', '4.1.2']

OK   P1 Q4 sub='4.1'     → 6 rows
OK   P1 Q4 sub='4.2'     → 6 rows
OK   P1 Q5 sub='5'       → 5 rows
OK   P1 Q8 sub='8'       → 7 rows
OK   P1 Q9 sub='9'       → 1 rows
OK   P1 Q10 sub='10'      → 7 rows
OK   P2 Q1 sub='1'       → 7 rows
OK   P2 Q2 sub='2'       → 5 rows
OK   P2 Q3 sub='3'       → 8 rows
OK   P2 Q4 sub='4'       → 7 rows
OK   P2 Q6 sub='6'       → 7 rows
OK   P2 Q7 sub='7'       → 3 rows
OK   P2 Q8 sub='8'       → 7 rows
OK   P2 Q9 sub='9.2'     → 2 rows
OK   P2 Q10 sub='10'      → 3 rows

Rows updated with vision supplement: 81
Unmatched vision rows: 0

--- extraction_method ---
extraction_method
vision_supplement    81
ocr_text             35
Name: count, dtype: int64

--- vision_verification_status ---
vis

In [18]:
from pathlib import Path
import pandas as pd

ROOT = Path(r'C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence')
MAP  = ROOT / "data" / "processed" / "mapped" / "2022" / "question_topic_map_2022.csv"

mapped = pd.read_csv(MAP)

# Rows where the diagram is LOAD-BEARING (cannot answer from text alone).
# Everything else with vision_supplement reverts to ocr_text but keeps
# diagram_present = True if it was in the original diagram_present set.
LOAD_BEARING = {
    # (paper, subquestion)  — the specific subquestions where the diagram
    # is required to determine the answer
    ("P1", "4.1.1"), ("P1", "4.1.2"), ("P1", "4.1.3"), ("P1", "4.1.4"), ("P1", "4.1.5"),
    ("P1", "4.2.1"), ("P1", "4.2.2"), ("P1", "4.2.3"), ("P1", "4.2.4"), ("P1", "4.2.5"),
    ("P1", "5.3"),   ("P1", "5.5"),
    ("P1", "8.1"),   ("P1", "8.2.1"), ("P1", "8.2.2"), ("P1", "8.3.1"), ("P1", "8.3.2"),
    ("P1", "9"),
    ("P1", "10.1.1"),("P1", "10.1.2"),("P1", "10.1.3"),("P1", "10.2.1"),("P1", "10.2.2"),
    ("P2", "1.1"),   ("P2", "1.2"),   ("P2", "1.3"),   ("P2", "1.4"),
    ("P2", "1.5.1"), ("P2", "1.5.2"),
    ("P2", "2.1"),   ("P2", "2.2"),   ("P2", "2.3"),   ("P2", "2.4"),   ("P2", "2.5"),
    ("P2", "3.1.1"), ("P2", "3.1.2"), ("P2", "3.1.3"), ("P2", "3.1.4"),
    ("P2", "3.2"),   ("P2", "3.3.1"), ("P2", "3.3.2"),
    ("P2", "4.1"),   ("P2", "4.2.1"), ("P2", "4.2.2"), ("P2", "4.3"),
    ("P2", "4.4.1"), ("P2", "4.4.2"),
    ("P2", "6.1"),   ("P2", "6.2.1"), ("P2", "6.2.2"), ("P2", "6.3"),
    ("P2", "6.4"),   ("P2", "6.5"),
    ("P2", "7.1"),   ("P2", "7.2"),   ("P2", "7.3"),
    ("P2", "8.1.1"), ("P2", "8.1.2"), ("P2", "8.1.3"),
    ("P2", "8.2.1"), ("P2", "8.2.2"),
    ("P2", "9.1"),   ("P2", "9.2.1"), ("P2", "9.2.2"),
    ("P2", "10.1"),  ("P2", "10.2"),  ("P2", "10.3"),
}

# Preserve the original diagram_present flag before reverting anything
if "diagram_present" not in mapped.columns:
    mapped["diagram_present"] = False

# Rows currently tagged vision_supplement
currently_vision = mapped["extraction_method"] == "vision_supplement"

# Of those, which are actually load-bearing?
def is_load_bearing(row):
    key = (row["paper"], str(row["subquestion"]).strip())
    return key in LOAD_BEARING

load_bearing_mask = currently_vision & mapped.apply(is_load_bearing, axis=1)

# Rows that were over-tagged: keep diagram_present = True, revert extraction_method
over_tagged = currently_vision & ~load_bearing_mask

mapped.loc[load_bearing_mask, "diagram_present"] = True
mapped.loc[over_tagged, "diagram_present"] = True
mapped.loc[over_tagged, "extraction_method"] = "ocr_text"
mapped.loc[over_tagged, "vision_description"] = None
mapped.loc[over_tagged, "vision_verification_status"] = None

mapped.to_csv(MAP, index=False)

print("--- after correction ---")
print(mapped["extraction_method"].value_counts())
print()
print(mapped["diagram_present"].value_counts())
print()
print("Rows reverted to ocr_text:", int(over_tagged.sum()))
print("Rows still tagged vision_supplement:", int(load_bearing_mask.sum()))

--- after correction ---
extraction_method
vision_supplement    63
ocr_text             53
Name: count, dtype: int64

diagram_present
True     82
False    34
Name: count, dtype: int64

Rows reverted to ocr_text: 18
Rows still tagged vision_supplement: 63


In [19]:
from pathlib import Path
import pandas as pd

ROOT = Path(r'C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence')
MAP  = ROOT / "data" / "processed" / "mapped" / "2022" / "question_topic_map_2022.csv"

mapped = pd.read_csv(MAP)

# Strict load-bearing set: answer CANNOT be determined from text alone.
LOAD_BEARING = {
    # P1 — Venn diagram only
    ("P1", "10.1.1"), ("P1", "10.1.2"), ("P1", "10.1.3"),
    # P2 — Scatter plot interpretation
    ("P2", "1.5.1"),
    # P2 — Ogive readings
    ("P2", "2.1"), ("P2", "2.2"), ("P2", "2.3"), ("P2", "2.5"),
    # P2 — Euclidean geometry with angle labels only in diagram
    ("P2", "8.1.1"), ("P2", "8.1.2"), ("P2", "8.1.3"),
    ("P2", "8.2.1"), ("P2", "8.2.2"),
    # P2 — Euclidean proofs (diagram is the problem)
    ("P2", "9.1"), ("P2", "9.2.1"), ("P2", "9.2.2"),
    ("P2", "10.1"), ("P2", "10.2"), ("P2", "10.3"),
}

def is_lb(row):
    return (row["paper"], str(row["subquestion"]).strip()) in LOAD_BEARING

# Reset vision fields
mapped["diagram_present"] = mapped["diagram_present"].fillna(False)
mapped["extraction_method"] = "ocr_text"
mapped["vision_description"] = None
mapped["vision_verification_status"] = None

# Now apply strictly
lb_mask = mapped.apply(is_lb, axis=1)
mapped.loc[lb_mask, "extraction_method"] = "vision_supplement"
mapped.loc[lb_mask, "vision_verification_status"] = "ok_verified_2026_09_18"
mapped.loc[lb_mask, "diagram_present"] = True

# Any subquestion in a diagram-question gets diagram_present = True
# even if not load-bearing — the diagram is present, just not required.
DIAGRAM_QUESTIONS = {
    ("P1", "4"), ("P1", "5"), ("P1", "8"), ("P1", "9"), ("P1", "10"),
    ("P2", "1"), ("P2", "2"), ("P2", "3"), ("P2", "4"), ("P2", "6"),
    ("P2", "7"), ("P2", "8"), ("P2", "9"), ("P2", "10"),
}
def in_diagram_q(row):
    qnum = str(row["question_number"]).strip().split(".")[0]
    return (row["paper"], qnum) in DIAGRAM_QUESTIONS

mapped.loc[mapped.apply(in_diagram_q, axis=1), "diagram_present"] = True

mapped.to_csv(MAP, index=False)

print("--- extraction_method ---")
print(mapped["extraction_method"].value_counts())
print()
print("--- diagram_present ---")
print(mapped["diagram_present"].value_counts())
print()
print("--- vision_verification_status ---")
print(mapped["vision_verification_status"].value_counts(dropna=False))

--- extraction_method ---
extraction_method
ocr_text             98
vision_supplement    18
Name: count, dtype: int64

--- diagram_present ---
diagram_present
True     82
False    34
Name: count, dtype: int64

--- vision_verification_status ---
vision_verification_status
None                      98
ok_verified_2026_09_18    18
Name: count, dtype: int64


In [20]:
from pathlib import Path
import pandas as pd

ROOT = Path(r'C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence')
OUT = ROOT / "docs" / "pilot" / "2023_2025"
OUT.mkdir(parents=True, exist_ok=True)

issues = [
    {"year": 2025, "paper": "P1", "subq": "9.4", "issue": "-1lx should be -11x", "severity": "Low"},
    {"year": 2025, "paper": "P1", "subq": "4",   "issue": "log_1 x subscript lost", "severity": "Medium"},
    {"year": 2025, "paper": "P2", "subq": "9.2.1", "issue": "Q-hat Q not in diagram, likely L-hat", "severity": "Medium"},
    {"year": 2024, "paper": "P1", "subq": "5",   "issue": "Skctched typo", "severity": "Low"},
    {"year": 2024, "paper": "P2", "subq": "2",   "issue": "numbered 1.1 instead of 2.1", "severity": "Low"},
    {"year": 2024, "paper": "P2", "subq": "6.2", "issue": "square root symbol lost", "severity": "Medium"},
    {"year": 2023, "paper": "P2", "subq": "-",   "issue": "page 14 answer grid misread as 1s", "severity": "Low"},
]

# Load-bearing lists for backfill
load_bearing = {
    2023: {"P1": [], "P2": ["8.1","8.2","8.3.1","8.3.2","9.1","9.2","9.3","10.1","10.2","10.3"]},
    2024: {"P1": [], "P2": ["2.1","2.2","2.3","2.4","2.5","7.4",
                            "9.1","9.2","10.1","10.2.1","10.2.2",
                            "11.1","11.2","11.3","11.4"]},
    2025: {"P1": ["9.3"], "P2": ["9.1","9.2.1","9.2.2",
                                  "10.1","10.2","10.3",
                                  "11.1.1","11.1.2","11.1.3",
                                  "11.2"]},
}

pd.DataFrame(issues).to_csv(OUT / "ocr_issues_2023_2025.csv", index=False)

lb_rows = []
for year, papers in load_bearing.items():
    for paper, subs in papers.items():
        for s in subs:
            lb_rows.append({"year": year, "paper": paper, "subquestion": s})
pd.DataFrame(lb_rows).to_csv(OUT / "load_bearing_2023_2025.csv", index=False)

print(f"Wrote {OUT}/ocr_issues_2023_2025.csv ({len(issues)} rows)")
print(f"Wrote {OUT}/load_bearing_2023_2025.csv ({len(lb_rows)} rows)")

Wrote C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\docs\pilot\2023_2025/ocr_issues_2023_2025.csv (7 rows)
Wrote C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\docs\pilot\2023_2025/load_bearing_2023_2025.csv (36 rows)
